In [1]:
"""keep in mind that when we actually do this we need to get rid of the _toy thing
 because we wont need that later"""

'keep in mind that when we actually do this we need to get rid of the _toy thing\n because we wont need that later'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

def protein_id_from_npz(npz_path):
    return npz_path.stem.replace("_toy", "")   # "seo_5_toy" -> "seo_5"

def numeric_id(protein_id):
    return int(protein_id.split("_")[1])       # "seo_5" -> 5

# --- Step 1: load labels (all columns, string IDs) ---
labels = pd.read_csv("seo_tested_seq_sorted_toy.csv")
labels["Library ID"] = labels["Library ID"].astype(str)
labels["_sort_key"] = labels["Library ID"].map(numeric_id)
labels = labels.sort_values("_sort_key").drop(columns="_sort_key").reset_index(drop=True)

# --- Step 2: load embeddings, mean-pool each protein (numeric order) ---
emb_dir = Path("embeddings")
ids = []
X_list = []

for npz_path in sorted(emb_dir.glob("seo_*_toy.npz"), key=lambda p: numeric_id(protein_id_from_npz(p))):
    protein_id = protein_id_from_npz(npz_path)
    emb = np.load(npz_path)["A_embeddings"]          # (270, 1280)
    protein_vec = emb.mean(axis=0)                   # (1280,)
    ids.append(protein_id)
    X_list.append(protein_vec)

X = np.stack(X_list)        # shape (5, 1280) — features
ids = np.array(ids)         # ["seo_5", "seo_8", "seo_11", "seo_13", "seo_19"]

# --- Step 3: align labels to same order as X ---
meta = labels.set_index("Library ID").loc[ids].reset_index()  # index -> column, no 0,1,2... left over
y = meta["Activity"].values   # shape (5,) — labels

# --- sanity check ---
print("X:", X.shape)
print("y:", y.shape)
print(meta[["Library ID", "Activity"]].to_string(index=False))

X: (5, 1280)
y: (5,)
Library ID   Activity
     seo_5  25.691800
     seo_8  44.146114
    seo_11  22.424004
    seo_13   9.636493
    seo_19 248.167811


In [3]:
"""this cells simply imports the toolkit, creates the model object ridge, and tells it to fit (X,y), or
in other words it trains on all 5 data points. We didn't specifiy a train/test split, which means we
also didn't do cross fold validation"""

# from sklearn.linear_model import Ridge

# model = Ridge()
# model.fit(X, y)

# predictions = model.predict(X)
# print(predictions)
# print(y)  # compare to true activities

"this cells simply imports the toolkit, creates the model object ridge, and tells it to fit (X,y), or\nin other words it trains on all 5 data points. We didn't specifiy a train/test split, which means we\nalso didn't do cross fold validation"

In [4]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

model = Ridge()

# 5-fold CV: each protein gets held out once (with only 5 samples)
scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print(scores)       # one score per fold
print(scores.mean())  # average

[nan nan nan nan nan]
nan


/opt/miniconda3/envs/petase_structure_activity/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/miniconda3/envs/petase_structure_activity/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/miniconda3/envs/petase_structure_activity/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/miniconda3/envs/petase_structure_activity/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/miniconda3/envs/petase_structure_ac